In [ ]:
# %uv add numpy matplotlib Ipython ipywidgets

# The mathematics of the model

The neuron model is as follows. There is a single variable $V$ for the neuron, its membrane potential (the potential difference between the inside and outside of the cell). In the absence of any input, the variable evolves over time according to the differential equation

$$
\tau \frac{\mathrm{d}V}{\mathrm{d}t} = -V.
$$

Here the constant $\tau$ is the time constant of the neuron, and controls how quickly the neuron integrates its inputs. You can easily solve this differential equation to get

$$
V(t) = V(0) e^{-t/\tau}.
$$

In other words, $V$ exponentially decays to the value 0, and the smaller $\tau$ is the faster it does it. You can verify this with the interactive widget below.

If the neuron receives an incoming spike with synaptic weight $w$, the membrane potential instantly increases by $w$, i.e.

$$V\leftarrow V+w.$$

In the interactive widget below, incoming spikes arrive at times $t_0$, $t_1$ and $t_2$, which you can change to see the effect.

If one of the incoming spikes causes $V$ to cross a *threshold* value, sometimes written as $V_t$, then the neuron will fire a spike, and instantaneously reset to the *reset* value $V_r$. In equations:

$$
\text{If } V>V_t \text{ then fire a spike and } V\leftarrow V_r.
$$

The net effect of all of this is that when $\tau$ is large, the neuron acts as an *integrator*, summing its inputs and firing when they reach some threshold. If $\tau$ is small, the neuron acts as a *coincidence detector*, firing a spike only if two or more inputs arrive simultaneously.

In [ ]:
try:
    import ipywidgets as widgets # type: ignore

except ImportError:
    widgets = None

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:

fig = plt.figure(figsize=(5, 4))
ax = plt.subplot(111)
plt.close(fig)

# function that runs the simulation
# tau = time constant (in ms)
# t0 , t1 t2 are the times of three spikes
# W = input synapse weight
# threshold = threshold value to produce a spike
#reset = reset value after spike

def LIF(tau=10, t0=20, t1=40, t2=60, w=0.1, threshold=1.0, reset=0.0):
    #spike times, keep sorted because it's more efficient to pop last value of the list
    times = [t0, t1, t2]
    times.sort(reverse=True)
    #set some default parameters
    duration = 100 #total times in ms
    dt = 0.1 #timesteps in ms
    alpha = np.exp(-dt/tau) # this is the factor by which V decays at each time step
    V_rec = [] # list to record membrane potentiol
    V = 0.0 # Initial membrane potential
    T = np.arange(np.round(duration/dt))*dt # array of time
    spikes = []
    #running the simulation
    for t in T:
        V_rec.append(V) #record
        V *= alpha
        if times and t > times[-1]:
            V += w
            times.pop()

        V_rec.append(V)
        if V>threshold:
            V = reset
            spikes.append(t)

    ax.clear()
    for t in times:
        ax.axvline(t, ls=':', c='b')
    ax.plot(np.repeat(T, 2), V_rec, '-k', lw=2)
    for t in spikes:
        ax.axvline(t, ls='--', c='r')
    ax.axhline(threshold, ls='--', c='g')
    ax.set_xlim(0, duration)
    ax.set_ylim(-1, 2)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('V')
    plt.tight_layout()
    display(fig);
    
# Create an interactive widget
widgets.interact(LIF, # type: ignore
    tau=widgets.IntSlider(min=1, max=100, value=50), # type: ignore
    t0=widgets.IntSlider(min=0, max=100, value=20), # type: ignore
    t1=widgets.IntSlider(min=0, max=100, value=40), # type: ignore
    t2=widgets.IntSlider(min=0, max=100, value=60), # type: ignore
    w=widgets.FloatSlider(min=-1, max=2, step=0.05, value=0.5), # type: ignore 
    threshold=widgets.FloatSlider(min=0.0, max=2.0, step=0.05, value=1.0), # type: ignore
    reset=widgets.FloatSlider(min=-1.0, max=1.0, step=0.05, value=0.0), # type: ignore
    );
        